In [1]:
import numpy as np

rng = np.random.default_rng(3)
fs = 5000                 # 샘플링 주파수(Hz)
N = fs                    # 1초 윈도우
t = np.arange(N) / fs

def make_window(anomaly=False, severity=1.0):
    x = (1.0 * np.sin(2*np.pi*30*t + rng.uniform(0, 2*np.pi))
         + 0.3 * np.sin(2*np.pi*60*t + rng.uniform(0, 2*np.pi))
         + rng.normal(0, 0.3, N))
    if anomaly:
        imp = np.zeros(N)
        imp[::int(fs / 157)] = 1                         # 157Hz 주기 충격
        k = np.arange(200)
        ring = np.exp(-k / 20) * np.sin(2*np.pi*2000*k/fs)   # 2kHz 공진 감쇠파
        x = x + severity * 0.8 * np.convolve(imp, ring, mode='full')[:N]
    return x

train       = np.array([make_window() for _ in range(300)])                    # 정상만
test_normal = np.array([make_window() for _ in range(200)])
test_anom   = np.array([make_window(True, s) for s in np.linspace(0.2, 1.5, 30)])  # 주입 이상

np.savez('/content/vibration.npz', train=train, test_normal=test_normal,test_anom=test_anom, fs=fs)